# Buổi 2 — Phân phối, Khoảng tin cậy, Kiểm định giả thuyết (Bài 4, 5, 7)
Ý tưởng chủ đạo: **không học thuộc công thức — tự mô phỏng để thấy p-value và khoảng tin cậy là gì.**

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10   # đã xử lý ngoại lai ở buổi 1
df["gain"] = df.posttest - df.pretest

## 1. Định lý giới hạn trung tâm (CLT) — vì sao phân phối chuẩn xuất hiện khắp nơi
Dữ liệu gốc lệch phải (`study_hours`), nhưng **trung bình mẫu** lại gần chuẩn khi n tăng.

In [ ]:
rng = np.random.default_rng(1)
pop = df.study_hours.to_numpy()
fig, ax = plt.subplots(1, 4, figsize=(14, 2.8), sharex=True)
sns.histplot(pop, bins=25, ax=ax[0]); ax[0].set_title("Dữ liệu gốc (lệch)")
for a, n in zip(ax[1:], [5, 30, 100]):
    means = [rng.choice(pop, n).mean() for _ in range(3000)]
    sns.histplot(means, bins=25, ax=a); a.set_title(f"Trung bình mẫu, n={n}\nSE≈{np.std(means):.2f}")
plt.tight_layout(); plt.show()
print("SE lý thuyết = s/√n với n=30:", round(pop.std(ddof=1) / np.sqrt(30), 3))

**❓** Khi n tăng 4 lần, sai số chuẩn (SE) giảm bao nhiêu lần? Suy ra: muốn độ chính xác gấp đôi cần bao nhiêu dữ liệu?

## 2. Khoảng tin cậy 95% nghĩa là gì?
Mô phỏng: lấy 1000 mẫu, dựng 1000 khoảng tin cậy; đếm bao nhiêu khoảng chứa trung bình thật.

In [ ]:
mu = pop.mean(); n = 30; hit = 0; los = []
for _ in range(1000):
    s = rng.choice(pop, n); m, se = s.mean(), stats.sem(s)
    lo, hi = stats.t.interval(0.95, n - 1, loc=m, scale=se); hit += lo <= mu <= hi; los.append((lo, hi))
print(f"Tỉ lệ khoảng chứa μ thật: {hit/1000:.1%}  (kỳ vọng ≈ 95%)")
plt.figure(figsize=(7, 4))
for i, (lo, hi) in enumerate(los[:40]):
    plt.plot([lo, hi], [i, i], color="C0" if lo <= mu <= hi else "C3")
plt.axvline(mu, color="k", ls="--"); plt.title("40 khoảng tin cậy đầu — đỏ = trượt μ"); plt.yticks([]); plt.show()

> **Hiểu đúng:** "Nếu lặp quy trình này nhiều lần, ~95% khoảng dựng ra sẽ chứa giá trị thật." KHÔNG phải "xác suất μ nằm trong khoảng này là 95%" (μ là một hằng số).

## 3. Logic của kiểm định giả thuyết & p-value
Đặt H0: *hai phương pháp dạy không khác nhau*. p-value = xác suất thấy chênh lệch **cực đoan như dữ liệu**, *nếu H0 đúng*.

In [ ]:
# 3a. Mô phỏng thế giới H0 đúng: 5000 lần so sánh hai nhóm cùng phân phối
ps = [stats.ttest_ind(rng.normal(0, 1, 30), rng.normal(0, 1, 30)).pvalue for _ in range(5000)]
print("Tỉ lệ p<0.05 khi H0 ĐÚNG (sai lầm loại I):", np.mean(np.array(ps) < .05))
sns.histplot(ps, bins=20); plt.title("p-value khi H0 đúng: phân bố ĐỀU"); plt.show()

In [ ]:
# 3b. Sức mạnh kiểm định (power) — sai lầm loại II
def power(d, n, sims=3000):
    return np.mean([stats.ttest_ind(rng.normal(0, 1, n), rng.normal(d, 1, n)).pvalue < .05 for _ in range(sims)])
print(pd.DataFrame({n: [power(d, n) for d in (0.2, 0.5, 0.8)] for n in (20, 50, 100, 200)}, index=["d=0.2", "d=0.5", "d=0.8"]).round(2))

**❓** Nhóm chỉ 20 em, hiệu ứng thật vừa (d=0.5): khả năng phát hiện ra nó là bao nhiêu? Nếu "không có ý nghĩa" thì có kết luận được phương pháp vô hiệu không?

## 4. T-test độc lập & bắt cặp trên dữ liệu lớp
SPSS: `Analyze > Compare Means > Independent-Samples T Test` / `Paired-Samples T Test`.

In [ ]:
a = df[df.method == "Dự án"].gain; b = df[df.method == "Truyền thống"].gain
print("Levene (phương sai bằng nhau?):", stats.levene(a, b))
t, p = stats.ttest_ind(a, b, equal_var=False)          # Welch — mặc định an toàn
d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
ci = stats.t.interval(.95, len(a) + len(b) - 2, loc=a.mean() - b.mean(), scale=np.sqrt(a.var()/len(a) + b.var()/len(b)))
print(f"Chênh lệch tăng điểm = {a.mean()-b.mean():.2f}  t={t:.2f}  p={p:.3g}  Cohen d={d:.2f}  CI95=({ci[0]:.2f},{ci[1]:.2f})")

In [ ]:
# Tự tính t từ công thức để thấy không có phép màu
se = np.sqrt(a.var()/len(a) + b.var()/len(b)); print("t thủ công =", round((a.mean() - b.mean()) / se, 3))
# Bắt cặp: trước-sau trên cùng học sinh
print(stats.ttest_rel(df.posttest, df.pretest)); print("Tương đương one-sample trên hiệu:", stats.ttest_1samp(df.gain, 0))

**❓ Thảo luận nghiên cứu giáo dục**
1. p<0.05 nhưng d nhỏ (~0.1) trên mẫu 10.000 em: có "ý nghĩa thống kê" — có "ý nghĩa thực tiễn"?
2. Vì sao dùng t-test **bắt cặp** cho trước–sau thay vì độc lập? (gợi ý: điểm trước và sau của cùng em tương quan cao → phương sai của hiệu nhỏ đi)
3. Ở đây học sinh *không* được gán ngẫu nhiên vào phương pháp thì kết luận "phương pháp dự án hiệu quả hơn" có nhân quả được không?

## 5. Bài tập
1. So sánh `math` giữa Nam–Nữ: đủ 4 thứ (Levene, t, d, CI). Viết kết luận 3 câu kiểu APA.
2. Chạy lại `power()` để tìm n cần thiết cho power 80% khi d=0.4.
3. Đổi seed trong mô phỏng CI: tỉ lệ có đúng 95% không? Vì sao dao động?